<a href="https://colab.research.google.com/github/ksw9179/AI_and_Data-/blob/main/RMSD%EA%B0%9C%EC%84%A0%EC%95%88_%EB%8B%A8%EB%B0%B1%EC%A7%88_3%EC%B0%A8%EC%9B%90_%EA%B5%AC%EC%A1%B0_%EB%B0%8F_%EC%9D%B8%ED%84%B0%EB%9E%99%EC%85%98_%EB%94%A5%EB%9F%AC%EB%8B%9D_%EC%98%88%EC%B8%A1%5BTop%5D_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein 3차원(3D) 구조 예측 및 3D 모형 유사도(RMSD) 계산

In [ ]:
# 1. 3D 시각화 및 생물학 데이터 분석을 위한 필수 패키지 설치
!pip install py3Dmol biopython requests -q

import requests
import py3Dmol
import io
from Bio.PDB import PDBParser
import time

In [ ]:
print("🧬 [Stage 3 확장] 여러 단백질의 3차원 구조 동시 예측 및 분석 시작...")

# 2. 전체 서열 데이터 베이스 - Stage 2에서 썼던 단백질 이름과 서열 묶음 (사전 데이터 구조)
protein_db = {
    "HER2": "MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPINCTHSCVDLDDKGCPAEQRASPLTSIISAVVGILLVVVLGVVFGILIKRRQQKIRKYTMRRLLQETELVEPLTPSGAMPNQAQMRILKETELRKVKVLGSGAFGTVYKGIWIPDGENVKIPVAIKVLRENTSPKANKEILDEAYVMAGVGSPYVSRLLGICLTSTVQLVTQLMPYGCLLDHVRENRGRLGSQDLLNWCMQIAKGMSYLEDVRLVHRDLAARNVLVKSPNHVKITDFGLARLLDIDETEYHADGGKVPIKWMALESILRRRFTHQSDVWSYGVTVWELMTFGAKPYDGIPAREIPDLLEKGERLPQPPICTIDVYMIMVKCWMIDSECRPRFRELVSEFSRMARDPQRFVVIQNEDLGPASPLDSTFYRSLLEDDDMGDLVDAEEYLVPQQGFFCPDPAPGAGGMVHHRHRSSSTRSGGGDLTLGLEPSEEEAPRSPLAPSEGAGSDVFDGDLGMGAAKGLQSLPTHDPSPLQRYSEDPTVPLPSETDGYVAPLTCSPQPEYVNQPDVRPQPPSPREGPLPAARPAGATLERPKTLSPGKNGVVKDVFAFGGAVENPEYLTPQGGAAPQPHPPPAFSPAFDNLYYWDQDPPERGAPPSTFKGTPTAENPEYLGLDVPV",
    "EGFR": "MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCLVCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEGPCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLFGTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSCRNVSRGRECVDKCNLLEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGVMGENNTLVWKYADAGHVCHLCHPNCTYGCTGPGLEGCPTNGPKIPSIATGMVGALLLLLVVALGIGLFMRRRHIVRKRTLRRLLQERELVEPLTPSGEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQGFFSSPSTSRTPLLSSLSATSNNSTVACIDRNGLQSCPIKEDSFLQRYSSDPTGALTEDSIDDTFLPVPEYINQSVPKRPAGSVQNPVYHNQPLNPAPSRDPHYQDPHSTAVGNPEYLNTVQPTCVNSTFDSPAHWAQKGSHQISLDNPDYQQDFFPKEAKPNGIFKGSTAENAEYLRVAPQSSEFIGA",
    "PDL1_1": "MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYNKINQRILVVDPVTSEHELTCQAEGYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTNEIFYCTFRRLDPEENHTAELVIPELPLAHPPNERTHLVILGAILLCLGVALTFIFRLRKGRMMDVKKCGIQDTNSKKQSDTHLEET"
}

# 도면 해독기(Parser) 미리 대기시키기
parser = PDBParser(QUIET=True)

In [ ]:
import io
from Bio.PDB import PDBParser, Superimposer

def calculate_structure_similarity(pdb_data1, pdb_data2, name1="Protein1", name2="Protein2"):
    """
    두 단백질 PDB 데이터 간의 CA 뼈대 좌표를 추출하여 구조적 유사도(RMSD)를 측정하는 함수
    """
    parser = PDBParser(QUIET=True)

    # 설계도 읽기
    struct1 = parser.get_structure(name1, io.StringIO(pdb_data1))
    struct2 = parser.get_structure(name2, io.StringIO(pdb_data2))

    # 핵심 척추 뼈대인 'CA' 원자들의 3D 좌표 정보만 쏙쏙 골라오기
    atoms1 = [atom for atom in struct1.get_atoms() if atom.get_name() == 'CA']
    atoms2 = [atom for atom in struct2.get_atoms() if atom.get_name() == 'CA']

    # 두 조각의 길이를 똑같이 맞추기
    min_length = min(len(atoms1), len(atoms2))
    fixed_atoms = atoms1[:min_length]
    moving_atoms = atoms2[:min_length]

    if min_length == 0:
        print(f"❌ [{name1} vs {name2}] 비교할 CA 원자 데이터가 부족합니다.")
        return None

    # 3D 공간에서 두 구조를 자석처럼 착! 겹치기
    imposer = Superimposer()
    imposer.set_atoms(fixed_atoms, moving_atoms)
    imposer.apply(moving_atoms)

    # 🔥 [수정 완료] 바이오파이썬 버전에 상관없이 오차를 가져오는 가장 안전한 변수명인 rms로 변경!
    rmsd_score = imposer.rms

    print(f"📊 [{name1} vs {name2}] 3D 구조 유사도 분석 결과")
    print(f"  - 비교에 사용된 중심 CA 원자 개수: {min_length}개")
    print(f"  - 뼈대 배치 오차 수치 (RMSD): {rmsd_score:.4f} Å")

    if rmsd_score < 1.5:
        print("  Interpretation: 두 단백질 조각은 전체적인 구조가 거의 완벽하게 일치합니다! (초고유사도)")
    elif rmsd_score < 3.0:
        print("  Interpretation: 두 단백질은 주요 뼈대와 도메인 구조가 매우 유사합니다.")
    else:
        print("  Interpretation: 두 단백질은 전체적인 3D 모양에 유의미한 차이가 있습니다.")

    return rmsd_score

In [ ]:
import os
import io
import time
import requests
import py3Dmol

# -------------------------------------------------------------
# [팁] 앞서 정의한 calculate_structure_similarity 함수 셀이
# 먼저 실행되어 있어야 유사도 분석 결과가 정상 출력됩니다!
# -------------------------------------------------------------

# 나중에 비교하기 위해 예측된 PDB 데이터를 안전하게 보관할 공간(딕셔너리)
predicted_pdbs = {}

# 3. 분석 시작
for p_name, full_seq in protein_db.items():
    print(f"\n🧪 [{p_name}] 단백질 3D 분석 시작...")
    num_chunks = (len(full_seq) // 400) + 1

    for i in range(num_chunks):
        chunk = full_seq[i*400 : (i+1)*400]
        name = f"{p_name}_Part_{i+1}"

        try:
            response = requests.post('https://api.esmatlas.com/foldSequence/v1/pdb/', data=chunk)
            if response.status_code == 200 and len(response.text) > 100:
                pdb_data = response.text

                # 🔥 [추가] 덮어쓰기 방지: 메모리 딕셔너리에 고유한 이름으로 각각 저장
                predicted_pdbs[name] = pdb_data

                # 🔥 [추가] 덮어쓰기 방지: /content 폴더에 진짜 PDB 파일로 곧바로 저장하기
                filename = f"/content/{name}.pdb"
                with open(filename, "w") as f:
                    f.write(pdb_data)
                print(f"💾 [{name}] 파일 저장 완료 -> {filename}")

                # 자신감 점수 계산
                structure = parser.get_structure(name, io.StringIO(pdb_data))
                scores = [atom.bfactor for atom in structure.get_atoms() if atom.bfactor is not None]
                avg_plddt = sum(scores) / len(scores) if scores else 0

                print(f"✅ [{name}] 예측 성공 | 신뢰도(pLDDT): {avg_plddt:.2f}/100")

                # [시각화 코드 부분]
                view = py3Dmol.view(width=600, height=400)
                view.addModel(pdb_data, 'pdb')

                # 보기 편한 무지개색(spectrum)으로 설정
                view.setStyle({'cartoon': {'color': 'spectrum'}})

                # 파트별로 '진짜 원래 번호'에 맞춰서 계산해서 빨간 점 찍기
                if p_name == "HER2":
                    # HER2과 항암제의 진짜 결합부위가 550~600번일 때
                    if i == 0: # Part_1 (1~400번 글자) -> 여기엔 550번이 없으므로 안 찍힘
                        view.zoomTo()
                        pass
                    elif i == 1: # Part_2 (401~800번 글자) -> 여기서는 150~200번이 됨!
                        view.addStyle({'resi': '150-200'}, {'sphere': {'color': 'red', 'scale': 1.5}})
                        view.zoomTo({'resi': '150-200'}) # 거기로 줌인

                elif p_name == "EGFR":
                    if i == 1: # EGFR도 Part_2에 결합 부위가 있다면 똑같이 처리
                        view.addStyle({'resi': '150-200'}, {'sphere': {'color': 'red', 'scale': 1.5}})
                        view.zoomTo({'resi': '150-200'})
                    else:
                        view.zoomTo()
                else:
                    view.zoomTo()
                view.show()

            else:
                 print(f"⚠️ [{name}] 예측 생략 (데이터 오류)")
        except:
            print(f"❌ [{name}] 분석 중 에러 발생")

        time.sleep(2) # 공장 서버 보호를 위한 매너 타임


# Protein 3D모형의 유사도(RMSD) 비교

In [ ]:
# -------------------------------------------------------------
# 🔥 [자동 분석] 모든 유전자의 파트별 분석 및 파일 저장이 완전히 끝난 뒤 실행!
# -------------------------------------------------------------
print("\n" + "="*60)
print("🧬 [최종 검증] 분리된 진짜 파일을 로드하여 3D 구조 유사도 비교")
print("="*60)

try:
    # 물리적인 파일 경로에서 진짜 각각의 데이터를 따로 읽어오기
    if os.path.exists("/content/HER2_Part_2.pdb") and os.path.exists("/content/EGFR_Part_2.pdb"):
        with open("/content/HER2_Part_2.pdb", "r") as f:
            real_her2_p2 = f.read()
        with open("/content/EGFR_Part_2.pdb", "r") as f:
            real_egfr_p2 = f.read()

        # 대문자 RMSD 에러가 수정된 우리의 계산기 실행!
        calculate_structure_similarity(real_her2_p2, real_egfr_p2, "HER2_Part_2", "EGFR_Part_2")
    else:
        print("⚠️ 비교할 HER2_Part_2.pdb 또는 EGFR_Part_2.pdb 파일이 폴더에 존재하지 않습니다.")
except NameError:
    print("❌ 에러: calculate_structure_similarity 함수가 정의된 셀을 먼저 실행해야 합니다.")

In [ ]:
import os
import io
from Bio.PDB import PDBParser, PDBIO

parser = PDBParser(QUIET=True)

print("⚙️ [독립형] PDB 파일 내부 아미노산 번호 진짜 번호로 업데이트 시작...")
print("="*60)

# 1. /content 폴더 안에 있는 모든 PDB 파일 가져오기
all_files = [f for f in os.listdir("/content") if f.endswith(".pdb")]

if not all_files:
    print("⚠️ 폴더에 .pdb 파일이 없습니다! 먼저 분석 엔진 코드를 실행해 주세요.")
else:
    for filename in sorted(all_files):
        # 파일 이름에서 'Part_2' 같은 글자를 보고 몇 번째 토막인지 알아내기
        # 예: Part_2 라면 part_num은 2가 됩니다.
        if "Part_" in filename:
            part_num = int(filename.split("Part_")[1].split(".pdb")[0])
        else:
            continue # Part 형식이 아니면 패스!

        # 🔥 [진짜 시작 번호 계산] Part_1이면 0, Part_2면 400, Part_3면 800을 더해줘요
        start_residue_offset = (part_num - 1) * 400

        file_path = os.path.join("/content", filename)

        # 파일 열어서 읽기
        with open(file_path, "r") as f:
            pdb_data = f.read()

        # 3D 구조 로봇으로 조립
        struct = parser.get_structure(filename, io.StringIO(pdb_data))

        # 🛠️ [번호판 교체] 아미노산 번호에 시작 오프셋 강제로 더하기
        for residue in list(struct.get_residues()):
            old_id = residue.id
            # 원래 1번이었던 번호에 400을 더해서 401번으로 바꿈!
            new_id = (old_id[0], old_id[1] + start_residue_offset, old_id[2])
            residue.id = new_id

        # 💾 진짜 번호로 바뀐 구조를 파일에 다시 덮어쓰기(저장)
        pdb_io = PDBIO()
        pdb_io.set_structure(struct)
        pdb_io.save(file_path)

        # 🚀 결과 출력 (선우가 확인하고 싶었던 진짜 범위!)
        residue_list = [res.id[1] for res in struct.get_residues()]
        print(f"🚀 {filename} ➡️ 진짜 아미노산 번호 범위: {min(residue_list)} ~ {max(residue_list)} 번 (총 {len(residue_list)}개)")

print("="*60)
print("✅ 모든 PDB 파일의 내부 번호판이 진짜 기차 번호로 영구 수정 및 저장되었습니다!")

# 3D모형간의 유사도(RMSM) 계산

# 🗺️ 2. 실제 Active Site(활성 부위)는 어떻게 찾아낼까? (정답 지도 찾기)
진짜 우리 몸속에서 거기가 활성 부위인지는 어떻게 알 수 있을까?

과학자들이 이미 수십 년 동안 실험해서 찾아낸 '정답 전사 데이터베이스'를 검색!

### ① 전 세계 단백질 백과사전: **UniProt (유니프롯)** 검색
설명: 전 세계 생물학자들이 단백질 하나를 분석할 때마다 그 비밀을 기록해 두는 '위키백과' 같은 사이트, UniProt(uniprot.org)

여기에 단백질 이름(예: P04626 - 인간 HER2의 고유 번호)을 검색하면,

Structure나 Function 탭에 "HER2의 700번부터 1000번 아미노산 자리가 ATP와 결합하는 핵심 활성 부위(Active Site)입니다" 라고 자로 잰 듯이 정답이 빽빽하게 적혀있음.

이걸 보고 우리 코드의 번호를 진짜 정답 번호로 수정

### ② 단백질 설계도 보관소: PDB (Protein Data Bank) 확인하기
설명: 다른 과학자가 이미 HER2 단백질에 실제 항암제 약을 찰떡같이 붙여놓고 X선 카메라로 찍어둔 3D 사진 파일이 PDB 데이터베이스에 수천 개나 저장되어 있음.

그 파일을 열어보면 "실제 약 분자가 HER2의 몇 번 아미노산 돌기에 대가리를 들이밀고 결합해 있구나" 하는 걸 눈으로 직접 확인할 수 있음.

그 달라붙어 있는 자리가 진짜 Active Site야.

### ③ 요즘 대세: AI에게 물어보기 (ScanNet, 딥러닝 예측)
설명: 만약 지구상에 아무도 실험 안 해본 완전히 새로운 단백질이라면 어떻게 할까?

요즘에는 ScanNet이나 DeepSite 같은 '활성부위 저격수 AI 모델'이 따로 있음.

이 AI한테 우리가 만든 3D 로봇(pdb_data)을 툭 던져주면,

AI가 표면의 구멍과 정전기 모양을 슥 훑어보고는

"주인님, 제가 보기에 이 단백질은 요 꼬불꼬불한 홈 구멍이 약이랑 결합하기 딱 좋은 조종석(Active Site) 같은데요?" 하고

확률 지도를 그려서 알려줌.

# 여기서부터는 target_sequence(타겟 단백질)이 한 개일 경우에만 코드 실행

In [ ]:
print("🧬 [Stage 3] HER2 단백질 3차원 구조 예측 및 분석 시작...")

# 2. Stage 2에서 사용했던 HER2 타겟 단백질 서열 (아미노산 알파벳)
target_sequence = "MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQ"

In [ ]:
# 3. 메타(Meta)의 ESMFold 인공지능 서버에 서열을 보내서 3D 구조(PDB) 받아오기
print("🤖 온라인 AI 구조 예측 공장(ESMFold)에 도면을 요청합니다...")
url = 'https://api.esmatlas.com/foldSequence/v1/pdb/'
response = requests.post(url, data=target_sequence)
pdb_data = response.text

In [ ]:
# 4. BioPython을 이용해 PDB 데이터 분석 및 AI의 자신감 점수(pLDDT) 계산
parser = PDBParser(QUIET=True)
structure = parser.get_structure("HER2_Target", io.StringIO(pdb_data))

# PDB 파일 안에는 AI가 얼마나 확신하는지(b-factor 자리에 pLDDT 점수)가 들어있음
plddt_scores = [atom.bfactor for atom in structure.get_atoms()]
average_plddt = sum(plddt_scores) / len(plddt_scores)

print(f"✅ AI의 구조 예측 평균 자신감 점수 (pLDDT): {average_plddt:.2f} / 100")
if average_plddt > 70:
    print("🌟 70점 이상이므로 AI가 이 단백질 구조를 아주 확신하고 있습니다!")

# 5. py3Dmol을 사용한 3차원 분자 구조 시각화 및 활성 부위(Active Site) 하이라이트
print("🔍 3D 뷰어를 생성합니다. 마우스로 드래그해서 돌려보세요!")
view = py3Dmol.view(width=800, height=500)

# 받아온 PDB 데이터를 뷰어에 추가
view.addModel(pdb_data, 'pdb')

# 단백질 전체를 무지개색 리본(Cartoon) 모양으로 표현
view.setStyle({'cartoon': {'color': 'spectrum'}})

# 면역 수용체와 결합할 것으로 예상되는 '핵심 활성 부위(아미노산 30번~50번)'를 빨간색 막대(Stick)로 강조
view.addStyle({'resi': '30-50'}, {'stick': {'colorscheme': 'redCarbon', 'radius': 0.2}})

# 화면 중앙으로 줌인 후 보여주기
view.zoomTo()
view.show()